
# **Riconoscimento di un logo con SIFT (Scale-Invariant Feature Transform)**

In questa esercitazione utilizzeremo **SIFT** per rilevare un logo all’interno di un’immagine.

Obiettivi:
1. Estrarre keypoint e descrittori SIFT.
2. Effettuare il matching tra il logo e la scena.
3. Localizzare il logo tramite omografia.


# **Import delle librerie**
È necessario eseguire l'import delle librerie utilizzate durante l'esercitazione.

In [ ]:
!pip install opencv-contrib-python matplotlib -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
# Funzione di utilità per visualizzare le immagini
def show_img(img, title=""):
    plt.figure(figsize=(8,6))
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

# **Caricamento e visualizzazione immagine logo e scena**


In [ ]:
logo_color = cv2.imread('logo.png')
scena_color = cv2.imread('scena.jpg')
show_img(logo_color)
show_img(scena_color)
logo = cv2.cvtColor(logo_color, cv2.COLOR_BGR2GRAY)
scena = cv2.cvtColor(scena_color, cv2.COLOR_BGR2GRAY)

# **Estrazione keypoint e descrittori locali e matching (con selezione dei good matches)**

In [ ]:
# Creazione dell'oggetto SIFT
sift = cv2.SIFT_create()

# Estrazione dei keypoint e descrittori
kp_logo, des_logo = sift.detectAndCompute(logo, None)
kp_scene, des_scene = sift.detectAndCompute(scena, None)

# Matching dei descrittori con BFMatcher (Brute Force)
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
matches = bf.knnMatch(des_logo, des_scene, k=2)

# Filtraggio dei match con il Lowe’s ratio test
good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

# Visualizzazione dei match
img_matches = cv2.drawMatches(logo_color, kp_logo, scena_color, kp_scene, good_matches, None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

show_img(img_matches, "Match SIFT")

# **Calcolo dell'omografia e visualizzazione del logo individuato nell'immagine della scena**

In [ ]:
# Calcolo dell'omografia per localizzare il logo
if len(good_matches) > 3:
    src_pts = np.float32([kp_logo[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 1.0)
    h, w = logo.shape
    pts = np.float32([[0, 0], [0, h-1], [w-1, h-1], [w-1, 0]]).reshape(-1, 1, 2)
    dst = cv2.perspectiveTransform(pts, M)

    img_result = cv2.polylines(scena_color,[np.int32(dst)], True, (0, 255, 0), 3, cv2.LINE_AA)

    show_img(img_result, "Logo individuato nella scena")
else:
    print("Troppi pochi match per calcolare l'omografia.")